# Notebook A - Tasks 1, 2, 3 (Oxford-IIIT Pet restoration)
Run on **Kaggle** (Settings -> Accelerator: **GPU T4 x2 or P100**, Internet: **On**) or Colab (Runtime -> T4 GPU).
Just run all cells. Total time with `BUDGET=fast` is roughly 1-1.5 h. Every stage is resumable: if the session dies, re-run and finished stages are skipped
(as long as the `outputs` folder still exists).
At the end download **outputs_A.zip** and send it back.

In [ ]:
# 1) Check GPU
!nvidia-smi
import os, sys
ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
ON_KAGGLE = (not ON_COLAB) and 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
WORK = '/kaggle/working' if ON_KAGGLE else '/content'
print('kaggle' if ON_KAGGLE else 'colab' if ON_COLAB else 'local', WORK)

In [ ]:
# 2) Get the code + install the few packages Kaggle/Colab do not have
os.chdir(WORK)
!rm -rf GenAI-Assignment1 && git clone -q https://github.com/maima2024/GenAI-Assignment1.git
os.chdir(os.path.join(WORK, 'GenAI-Assignment1'))
print(os.getcwd())
!pip install -q optuna mlflow onnx onnxruntime onnxscript
os.environ['BUDGET'] = 'fast'          # 'smoke' = 2-minute test, 'fast' = assignment run, 'full' = longer
os.environ['OUT_DIR'] = 'outputs'
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
os.environ['MLFLOW_DISABLE_AGENT_HINT'] = '1'

In [ ]:
# 3) Dataset: downloaded automatically with torchvision if not attached.
# On Kaggle you can alternatively 'Add Input' an Oxford-IIIT Pet dataset; its folder is searched automatically.
os.environ['PETS_SEARCH'] = '/kaggle/input' if ON_KAGGLE else ''
!python -c "import sys; sys.path.insert(0,'src'); from genai.common import get_budget; from genai.pet_data import setup_pets; d=setup_pets(get_budget()); print(d['trainval'].shape, d['test'].shape, len(d['train_idx']), len(d['val_idx']), len(d['test_manifest']))"

In [ ]:
# 4) Task 1 - universal denoising autoencoder (Optuna + final training + eval + ONNX)
!python scripts/task1_udae.py

In [ ]:
# 5) Task 2 - classifier + hard-routed specialists
!python scripts/task2_hard_routing.py

In [ ]:
# 6) Task 3 - soft mixture of experts
!python scripts/task3_soft_moe.py

In [ ]:
# 7) Package results (figures, metrics, MLflow runs, Optuna DB, ONNX models, checkpoints)
!cd outputs && zip -qr ../outputs_A.zip . -x '*test_per_item.pt' '*routing_per_item.pt'
!ls -lh outputs_A.zip outputs/onnx
if ON_KAGGLE:
    !cp outputs_A.zip /kaggle/working/outputs_A.zip
    from IPython.display import FileLink; display(FileLink('outputs_A.zip'))
elif ON_COLAB:
    from google.colab import files; files.download('outputs_A.zip')